# ASTR 350: Astronomical Techniques
# Lecture 1


### Prof. Robert Quimby <br> January 20, 2026

&copy; 2026 Robert Quimby

## What you do in other courses

![Blackbody curves showing Wien's displacement law](http://hyperphysics.phy-astr.gsu.edu/hbase/imgmod/wien3.gif)

#### Typical Question: what is the temperature of a star with $\lambda_{\rm peak}=500\,{\rm nm}$?

#### Typical Answer:

Use Wien's displacement law: $$T = {2.898 \times 10^{-3} m\,K \over \lambda_{\rm peak}}$$

In [ ]:
T = ????
print(f"The star's temperature is {round(T, -2):.0f} Kelvin")

## Example of what you might do in this course

There is a star at
$$
\begin{align}
\alpha && = && 17^{h}20^{m}39.57^{s} && \\
\delta && = && +32^{\circ}28^{\prime}03.9^{\prime\prime} && {\rm (J2000)}
\end{align}
$$

What is its temperature?

In [ ]:
# Celestial coordinates of the star
from astropy.coordinates import SkyCoord
pos = SkyCoord('17 20 39.57 +32 28 03.9', unit='hour,deg')

In [ ]:
# identify the target in Gaia
from astroquery.gaia import Gaia

query = f"""
SELECT source_id, phot_g_mean_mag AS g, 
  phot_bp_mean_mag AS g_bp, 
  phot_rp_mean_mag AS g_rp
FROM gaiadr3.gaia_source
WHERE 1=CONTAINS(
  POINT('ICRS', ra, dec),
  CIRCLE('ICRS', {pos.ra.deg}, {pos.dec.deg}, 0.01))
AND has_xp_sampled = 'True'
"""
job = Gaia.launch_job_async(query)
gaia = job.get_results()
gaia

In [ ]:
# retreive the Gaia spectra of the star
retrieval_type = 'XP_SAMPLED'
data_structure = 'INDIVIDUAL'
data_release   = 'Gaia DR3'
datalink = Gaia.load_data(ids=gaia['SOURCE_ID'], data_release=data_release,
                          retrieval_type=retrieval_type, data_structure=data_structure,
                          verbose=False, output_file=None)
dl_keys  = [inp for inp in datalink.keys()]
dl_key = dl_keys[0]
spec = datalink[dl_key][0].to_table()
spec

In [ ]:
# plot the flux densities
import matplotlib.pyplot as plt

plt.plot(spec['wavelength'], spec['flux'])
plt.xlabel('Wavelength (nm)')
plt.ylabel('Flux Density');
plt.grid();

In [ ]:
import numpy as np
import astropy.units as u
from astropy.modeling.models import BlackBody
def model_bb(wav, *params):
    temperature, scale = params
    bb = BlackBody(temperature * u.K)
    flux = bb(wav * u.nm) * (4 * np.pi * u.sr)
    flam = flux.to(u.W / u.m**2 / u.nm, 
                   equivalencies=u.spectral_density(wav * u.nm))
    return flam.value / scale / 1e6

# find the best-fitting model parameters
from scipy.optimize import curve_fit
guess = 6000, 1
popt, pcov = curve_fit(model_bb, spec['wavelength'], spec['flux'], p0=guess)

In [ ]:
# print the best-fit temperature
temp = popt[0]
etemp = np.sqrt(pcov[0, 0])
print(f"The best-fit temperature is {temp:.0f} +/- {etemp:.0f} Kelvin")

In [ ]:
# plot the data
plt.plot(spec['wavelength'], spec['flux'], label='Data')

# plot the best-fit model
wavs = np.linspace(300, 1200, 100)
model = model_bb(wavs, *popt)
plt.plot(wavs, model, label='BB model')

plt.xlabel('Wavelength (nm)')
plt.ylabel('Flux Density');
plt.legend()
plt.grid();

## Intellectual Skills (Critical Thinking)

![](https://upload.wikimedia.org/wikipedia/commons/thumb/6/6a/Bloom%27s_revised_taxonomy.svg/960px-Bloom%27s_revised_taxonomy.svg.png)

![](media/hardfonts.png)

see ["Fortune favors the **Bold** (*and the Italicized*): Effects of disfluency on educational outcomes"](https://www.sciencedirect.com/science/article/abs/pii/S001002771000226X) by Connor Diemand-Yauman, Daniel M. Oppenheimer, and Erikka B. Vaughan (2011)

## Be Wary of Chatbots

see ["Your brain on chatgpt: Accumulation of cognitive debt when using an ai assistant for essay writing task"](https://www.media.mit.edu/publications/your-brain-on-chatgpt/) by Kosmyna et al. (2025)

## AI policy

1. Only ask AI tools (e.g., chatbots) to give general explanations of concepts; never ask them your assignment prompts.
2. Never ask a chatbot to do anything that you would not ask a classmate or the professor to do. Remember, everything you submit must be your original work.
3. At the instructors discretion, any submitted assignment can be converted into an oral exam.

## Syllabus

[syllabus](https://github.com/SDSU-astr350-2026-spring/syllabus)

## Enrollment Form

- [Canvas](https://sdsu.instructure.com/courses/197577)
- [MyMap](https://sunspot.sdsu.edu/pubred/!mymap.disp) (SDSU's guide for completing your degree requirements in four years)

## GitHub

* I will use GitHub to distribute lecture notes and assignments
* Go to [GitHub.com](https://github.com/) and create an account for yourself if you don't already have one
* I will send you an invitation to join our class organization, https://github.com/SDSU-astr350-2026-spring

## Homework

[First homework assignment](../homework/astr350.hw01.ipynb)

## Tutorials

[list of tutorials](https://github.com/SDSU-astr350-2026-spring/tutorials)

## About your professor

![Headshot of Rober Quimby](media/quimby.png)

* BA, Astrophysics (UC Berkeley, 1998)
* MA, Astronomy (Texas, 2004)
* PhD, Astronomy (Texas, 2006)
* Postdoc, Caltech (2007-2011)
* Postdoc, Kavli IPMU (2011-2014)
* Joined SDSU faculty in Fall 2014
* MLO Director

* Research focus: Supernovae, transients

Research Highlights:
- Discovery of Superluminous Supernovae
- First identification of a gravitationally lensed Type Ia supernova